In [1]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORIES & COLOR SETUP
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Maps")
os.makedirs(output_folder, exist_ok=True) # Creates a new folder for your PNGs

years = [2000, 2005, 2010, 2015, 2020, 2025]

# LULC Setup
lulc_colors = ['#FF0000', '#FFA500', '#00A000', '#8B4513', '#0000FF']
lulc_cmap = ListedColormap(lulc_colors)
lulc_labels = ['Dense Urban', 'Sparse Urban', 'Vegetation', 'Bare Soil', 'Water Body']
lulc_patches = [mpatches.Patch(color=lulc_colors[i], label=lulc_labels[i]) for i in range(5)]

# UHI Setup
uhi_colors = ['#0000FF', '#FFFFFF', '#FF0000']
uhi_cmap = ListedColormap(uhi_colors)
uhi_labels = ['Coldspot (<-1.5)', 'Neutral (-1.5 to 1.5)', 'Hotspot (>1.5)']
uhi_patches = [mpatches.Patch(color=uhi_colors[i], label=uhi_labels[i]) for i in range(3)]

# ==========================================
# 2. MAP GENERATOR LOOP
# ==========================================
print("Generating high-resolution thesis maps...\n")

for year in years:
    file_path = os.path.join(base_folder, f"Enugu_{year}_COMPLETE_ANALYSIS_STACK.tif")
    
    with rasterio.open(file_path) as src:
        # Read only the required bands: 13 (UHI), 14 (LULC), 15 (Valid Mask)
        # Note: Rasterio uses 1-based indexing, so python indices 12, 13, 14 = Rasterio 13, 14, 15
        uhi_raw = src.read(13)
        lulc_raw = src.read(14)
        valid_mask = src.read(15)
        
    # Mask out dead space (NaNs) outside the city boundary
    city_mask = np.isnan(valid_mask)
    
    # Process LULC
    lulc_clean = np.ma.masked_where(city_mask, lulc_raw)
    
    # Process UHI (Apply your strict neural network classification rules)
    uhi_classes = np.zeros_like(uhi_raw)
    uhi_classes[uhi_raw < -1.5] = 0
    uhi_classes[(uhi_raw >= -1.5) & (uhi_raw <= 1.5)] = 1
    uhi_classes[uhi_raw > 1.5] = 2
    uhi_clean = np.ma.masked_where(city_mask, uhi_classes)
    
    # ==========================================
    # 3. PLOT AND EXPORT LULC
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    # Background color for NoData areas (outside city)
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(lulc_clean, cmap=lulc_cmap, vmin=0, vmax=4)
    ax.set_title(f"Land Use Land Cover (LULC) - Enugu Metropolis ({year})", fontsize=14, pad=15)
    ax.legend(handles=lulc_patches, loc='lower right', title="LULC Classes", facecolor='white', framealpha=1)
    ax.axis('off')
    
    lulc_output_path = os.path.join(output_folder, f"LULC_Map_{year}.png")
    plt.savefig(lulc_output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # ==========================================
    # 4. PLOT AND EXPORT UHI
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(uhi_clean, cmap=uhi_cmap, vmin=0, vmax=2)
    ax.set_title(f"Urban Heat Island (UHI) Distribution - Enugu Metropolis ({year})", fontsize=14, pad=15)
    ax.legend(handles=uhi_patches, loc='lower right', title="Thermal Intensity", facecolor='white', framealpha=1)
    ax.axis('off')
    
    uhi_output_path = os.path.join(output_folder, f"UHI_Map_{year}.png")
    plt.savefig(uhi_output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {year} maps.")

print(f"\nAll thesis maps saved to: {output_folder}")

Generating high-resolution thesis maps...

Successfully exported 2000 maps.
Successfully exported 2005 maps.
Successfully exported 2010 maps.
Successfully exported 2015 maps.
Successfully exported 2020 maps.
Successfully exported 2025 maps.

All thesis maps saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Maps


In [5]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORIES & COLOR SETUP
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Maps_V3")
os.makedirs(output_folder, exist_ok=True) 

years = [2000, 2005, 2010, 2015, 2020, 2024]

lulc_colors = ['#FF0000', '#FFA500', '#00A000', '#8B4513', '#0000FF']
lulc_cmap = ListedColormap(lulc_colors)
lulc_labels = ['Dense Urban', 'Sparse Urban', 'Vegetation', 'Bare Soil', 'Water Body']
lulc_patches = [mpatches.Patch(color=lulc_colors[i], label=lulc_labels[i]) for i in range(5)]

uhi_colors = ['#0000FF', '#FFFFFF', '#FF0000']
uhi_cmap = ListedColormap(uhi_colors)
uhi_labels = ['Coldspot (<-1.5)', 'Neutral (-1.5 to 1.5)', 'Hotspot (>1.5)']
uhi_patches = [mpatches.Patch(color=uhi_colors[i], label=uhi_labels[i]) for i in range(3)]

# ==========================================
# 2. MAP GENERATOR LOOP WITH COORDINATES
# ==========================================
print("Generating high-resolution thesis maps with corrected spatial masking...\n")

for year in years:
    file_path = os.path.join(base_folder, f"Enugu_{year}_COMPLETE_ANALYSIS_STACK.tif")
    
    with rasterio.open(file_path) as src:
        uhi_raw = src.read(13)
        lulc_raw = src.read(14)
        
        # Extract geographic bounding box for coordinates
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        
    # THE FIX: Use the raw thermal band to define the city boundary. 
    # If there is no temperature recorded, it is outside the city limits.
    city_mask = np.isnan(uhi_raw)
    
    # Apply the mask to LULC
    lulc_clean = np.ma.masked_where(city_mask, lulc_raw)
    
    # Apply strict classification rules to UHI, then mask
    uhi_classes = np.zeros_like(uhi_raw)
    uhi_classes[uhi_raw < -1.5] = 0
    uhi_classes[(uhi_raw >= -1.5) & (uhi_raw <= 1.5)] = 1
    uhi_classes[uhi_raw > 1.5] = 2
    uhi_clean = np.ma.masked_where(city_mask, uhi_classes)
    
    footer_text = "Project: Ocholi Israel MSc (Geoinformatics & Surveying, UNN) | Data Source: Landsat & Sentinel-2 (GEE)"
    
    # ==========================================
    # 3. PLOT AND EXPORT LULC
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(lulc_clean, cmap=lulc_cmap, vmin=0, vmax=4, extent=extent)
    ax.set_title(f"Land Use Land Cover (LULC) - Enugu Metropolis ({year})", fontsize=14, pad=15)
    
    ax.legend(handles=lulc_patches, loc='upper right', title="LULC Classes", facecolor='white', framealpha=1)
    ax.set_xlabel("Easting / Longitude", fontsize=10)
    ax.set_ylabel("Northing / Latitude", fontsize=10)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.grid(color='white', linestyle='--', alpha=0.4) 
    
    fig.text(0.5, 0.05, footer_text, ha='center', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    plt.savefig(os.path.join(output_folder, f"LULC_Map_{year}_Final.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # ==========================================
    # 4. PLOT AND EXPORT UHI
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E0E0E0') 
    
    ax.imshow(uhi_clean, cmap=uhi_cmap, vmin=0, vmax=2, extent=extent)
    ax.set_title(f"Urban Heat Island (UHI) Distribution - Enugu Metropolis ({year})", fontsize=14, pad=15)
    
    ax.legend(handles=uhi_patches, loc='upper right', title="Thermal Intensity", facecolor='white', framealpha=1)
    ax.set_xlabel("Easting / Longitude", fontsize=10)
    ax.set_ylabel("Northing / Latitude", fontsize=10)
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.grid(color='white', linestyle='--', alpha=0.4)
    
    fig.text(0.5, 0.05, footer_text, ha='center', fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    plt.savefig(os.path.join(output_folder, f"UHI_Map_{year}_Final.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {year} layout.")

print(f"\nFinal thesis maps saved to: {output_folder}")

Generating high-resolution thesis maps with corrected spatial masking...

Successfully exported 2000 layout.
Successfully exported 2005 layout.
Successfully exported 2010 layout.
Successfully exported 2015 layout.
Successfully exported 2020 layout.
Successfully exported 2024 layout.

Final thesis maps saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Maps_V3


In [4]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORY SETUP & INDEX CONFIGURATION
# ==========================================
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
output_folder = os.path.join(base_folder, "Thesis_Exported_Grids")
os.makedirs(output_folder, exist_ok=True) 

years = [2000, 2005, 2010, 2015, 2020, 2025]

# Define the exact file suffixes from your folder and assign scientific colormaps
indices = {
    'LST': {'suffix': 'LST_C', 'cmap': 'RdYlBu_r', 'title': 'Land Surface Temperature (°C)'},
    'NDVI': {'suffix': 'NDVI', 'cmap': 'RdYlGn', 'title': 'Normalized Difference Vegetation Index (NDVI)'},
    'NDBI': {'suffix': 'NDBI', 'cmap': 'YlOrBr', 'title': 'Normalized Difference Built-up Index (NDBI)'}
}

footer_text = "Project: Ocholi Israel MSc (Geoinformatics & Surveying, UNN) | Data Source: Landsat & Sentinel-2 (GEE)"

# ==========================================
# 2. GENERATE 2x3 GRIDS FOR EACH INDEX
# ==========================================
print("Compiling 2x3 Spatiotemporal Grids for Thesis Presentation...\n")

for index_name, props in indices.items():
    # Create a 2x3 grid of subplots that share the same X and Y coordinate axes
    fig, axes = plt.subplots(2, 3, figsize=(22, 14), sharex=True, sharey=True)
    fig.suptitle(f"Spatiotemporal Distribution of {props['title']} (2000 - 2025)", 
                 fontsize=22, fontweight='bold', y=0.96)
    
    # Flatten the 2x3 array of axes into a simple list for easy looping
    axes = axes.flatten()
    
    for idx, year in enumerate(years):
        file_path = os.path.join(base_folder, f"Enugu_{year}_{props['suffix']}.tif")
        
        try:
            with rasterio.open(file_path) as src:
                # Read the single band containing the index values
                data = src.read(1)
                
                # Extract coordinates for the axes
                bounds = src.bounds
                extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
                
            # Mask out the background outside the city boundary
            city_mask = np.isnan(data)
            clean_data = np.ma.masked_where(city_mask, data)
            
            ax = axes[idx]
            ax.set_facecolor('#E0E0E0') 
            
            # Render the map
            im = ax.imshow(clean_data, cmap=props['cmap'], extent=extent)
            ax.set_title(f"{year}", fontsize=18, pad=10, fontweight='bold')
            
            # Attach a colorbar to each individual year to accurately show changing max/min values
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=10)
            
            # Format Coordinates and Gridlines
            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.grid(color='white', linestyle='--', alpha=0.4)
            
            # Only add X-axis labels to the bottom row, and Y-axis labels to the left column
            if idx >= 3: 
                ax.set_xlabel("Easting / Longitude", fontsize=12)
            if idx % 3 == 0: 
                ax.set_ylabel("Northing / Latitude", fontsize=12)
                
        except Exception as e:
            print(f"Error loading {year} {index_name}: Ensure the file exists. ({e})")
            
    # Inject custom MSc Project attribution at the very bottom
    fig.text(0.5, 0.04, footer_text, ha='center', fontsize=14, 
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Adjust layout to prevent overlapping text and save
    plt.subplots_adjust(wspace=0.1, hspace=0.15)
    output_path = os.path.join(output_folder, f"{index_name}_Decadal_Grid.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Successfully exported {index_name} 2x3 Grid.")

print(f"\nAll index grids saved to: {output_folder}")

Compiling 2x3 Spatiotemporal Grids for Thesis Presentation...

Successfully exported LST 2x3 Grid.
Error loading 2025 NDVI: Ensure the file exists. (C:\Users\PC\Documents\MSc_Enugu_Data\Enugu_2025_NDVI.tif: No such file or directory)
Successfully exported NDVI 2x3 Grid.
Error loading 2025 NDBI: Ensure the file exists. (C:\Users\PC\Documents\MSc_Enugu_Data\Enugu_2025_NDBI.tif: No such file or directory)
Successfully exported NDBI 2x3 Grid.

All index grids saved to: C:\Users\PC\Documents\MSc_Enugu_Data\Thesis_Exported_Grids


In [6]:
import os
import numpy as np
import pandas as pd
import rasterio
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

print("--- EXTRACTING HISTORICAL LULC & UHI/LST STATISTICS ---")

base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
# Checking all potential methodology years; missing files will be safely skipped
years = [2000, 2005, 2010, 2015, 2020, 2024] 
TOTAL_CITY_AREA_KM2 = 560.39

# 1. Establish the authoritative city boundary mask from 2024
with rasterio.open(os.path.join(base_folder, "Enugu_2024_COMPLETE_ANALYSIS_STACK.tif")) as src:
    valid_mask_full = src.read(15)
    
inside_city_mask = ~np.isnan(valid_mask_full) & (valid_mask_full != 0)
total_valid_pixels = np.sum(inside_city_mask)
multiplier = TOTAL_CITY_AREA_KM2 / total_valid_pixels

# 2. Extract Data for Each Year
historical_stats_lulc = []
historical_stats_uhi = []

for year in years:
    file_path = os.path.join(base_folder, f"Enugu_{year}_COMPLETE_ANALYSIS_STACK.tif")
    
    if not os.path.exists(file_path):
        print(f"Skipping {year}: File not found.")
        continue
        
    print(f"Processing data for {year}...")
    with rasterio.open(file_path) as src:
        lulc_raw = src.read(14)
        lst_raw = src.read(13) # Assuming Band 13 is the LST/Thermal intensity channel
        
    # Apply spatial mask
    lulc_valid = lulc_raw[inside_city_mask]
    lst_valid = lst_raw[inside_city_mask]
    
    # Handle any internal NaNs
    lulc_valid = np.nan_to_num(lulc_valid, nan=2.0)
    lst_valid = np.nan_to_num(lst_valid, nan=np.nanmean(lst_valid))
    
    # Calculate LULC areas in km2
    dense = np.sum(lulc_valid == 0) * multiplier
    sparse = np.sum(lulc_valid == 1) * multiplier
    veg = np.sum(lulc_valid == 2) * multiplier
    bare = np.sum(lulc_valid == 3) * multiplier
    water = np.sum(lulc_valid == 4) * multiplier
    
    historical_stats_lulc.append({
        'Year': year,
        'Built-Up: Dense (km²)': round(dense, 2),
        'Built-Up: Sparse (km²)': round(sparse, 2),
        'Total Built-Up (km²)': round(dense + sparse, 2),
        'Vegetation (km²)': round(veg, 2),
        'Bare Soil (km²)': round(bare, 2),
        'Water (km²)': round(water, 2)
    })
    
    # Calculate UHI zones (km2) and Mean LST
    coldspot = np.sum(lst_valid < -1.5) * multiplier
    neutral = np.sum((lst_valid >= -1.5) & (lst_valid <= 1.5)) * multiplier
    hotspot = np.sum(lst_valid > 1.5) * multiplier
    mean_lst = np.nanmean(lst_valid)
    
    historical_stats_uhi.append({
        'Year': year,
        'Mean LST / Thermal Index': round(mean_lst, 3),
        'Coldspot Zone (km²)': round(coldspot, 2),
        'Neutral Zone (km²)': round(neutral, 2),
        'UHI Hotspot Zone (km²)': round(hotspot, 2)
    })

# Convert to Pandas DataFrames
df_hist_lulc = pd.DataFrame(historical_stats_lulc)
df_hist_uhi = pd.DataFrame(historical_stats_uhi)

# ==========================================
# 3. EXPORT TO MICROSOFT WORD
# ==========================================
print("\n--- GENERATING WORD DOCUMENT ---")
doc = Document()

# Set standard font
style = doc.styles['Normal']
style.font.name = 'Times New Roman'
style.font.size = Pt(12)

# Title
title = doc.add_heading('Historical Urban Development and UHI Statistics (2000–2024)', level=1)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph("This document details the historical area calculations for Land Use/Land Cover (LULC) transformations and Urban Heat Island (UHI) intensity zones in Enugu Metropolis, derived from multi-temporal Landsat imagery and spatial analysis.")

def add_dataframe_to_word(document, df, table_title):
    document.add_heading(table_title, level=2)
    table = document.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    
    # Add Header Row
    hdr_cells = table.rows[0].cells
    for i, column_name in enumerate(df.columns):
        hdr_cells[i].text = str(column_name)
        hdr_cells[i].paragraphs[0].runs[0].font.bold = True
        
    # Add Data Rows
    for index, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            row_cells[i].text = str(value)
            
    document.add_paragraph("\n")

# Insert the Tables
add_dataframe_to_word(doc, df_hist_lulc, "Table 1: Historical LULC Area Distribution (2000–2024)")
add_dataframe_to_word(doc, df_hist_uhi, "Table 2: Historical Mean LST and UHI Zone Distribution (2000–2024)")

# Save Document
output_word_path = os.path.join(base_folder, "Enugu_Historical_Stats_2000_2024.docx")
doc.save(output_word_path)

print(df_hist_lulc.to_string(index=False))
print("\n")
print(df_hist_uhi.to_string(index=False))
print(f"\n✅ Successfully exported historical statistical tables to: {output_word_path}")

--- EXTRACTING HISTORICAL LULC & UHI/LST STATISTICS ---
Processing data for 2000...
Processing data for 2005...
Processing data for 2010...
Processing data for 2015...
Processing data for 2020...
Processing data for 2024...

--- GENERATING WORD DOCUMENT ---
 Year  Built-Up: Dense (km²)  Built-Up: Sparse (km²)  Total Built-Up (km²)  Vegetation (km²)  Bare Soil (km²)  Water (km²)
 2000                  35.23                   12.20                 47.43            423.58            89.13         0.25
 2005                  71.43                  102.10                173.53            331.63            55.12         0.11
 2010                  34.07                    2.93                 36.99            471.97            51.20         0.23
 2015                  43.06                   12.04                 55.10            441.98            63.12         0.18
 2020                  84.02                   12.45                 96.47            345.16           118.59         0.18
 202

In [8]:
import os
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

print("--- FORMATTING CHAPTER 4 RESULTS TABLE ---")

# Define paths
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
csv_path = os.path.join(base_folder, "Enugu_LST_UHI_Stats_2000_2024.csv")
output_word_path = os.path.join(base_folder, "Chapter_4_UHI_LST_Table.docx")

# 1. Read and clean the data
df = pd.read_csv(csv_path)

# Filter out GEE metadata columns and order the relevant ones logically
columns_to_keep = ['Year', 'Mean LST (°C)', 'Coldspot Zone (km²)', 'Neutral Zone (km²)', 'UHI Hotspot Zone (km²)']
df = df[columns_to_keep]

# Sort by year ascending to show a chronological progression
df = df.sort_values(by='Year').reset_index(drop=True)

# 2. Create the Word Document
doc = Document()

# Set default font to Times New Roman, 12pt (Standard for academic dissertations)
style = doc.styles['Normal']
style.font.name = 'Times New Roman'
style.font.size = Pt(12)

# Add Chapter 4 Heading
doc.add_heading('4.x Historical Land Surface Temperature and UHI Distribution', level=2)
doc.add_paragraph(
    "Table X illustrates the decadal shifts in Mean Land Surface Temperature (LST) and the spatial extent of Urban Heat Island (UHI) intensity zones across Enugu Metropolis from 2000 to 2024. "
    "The physical area of each thermal zone (measured in km²) demonstrates the progressive conversion of cool vegetative boundaries into severe urban hotspots."
)

# Add Table Heading
table_heading = doc.add_paragraph()
table_heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = table_heading.add_run('Table X: Historical Mean LST and UHI Zone Distribution (2000–2024)')
run.bold = True

# 3. Construct the Word Table
table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'
table.autofit = True

# Add Header Row and make it bold
hdr_cells = table.rows[0].cells
for i, column_name in enumerate(df.columns):
    hdr_cells[i].text = str(column_name)
    # Apply bold formatting to headers
    for paragraph in hdr_cells[i].paragraphs:
        for run in paragraph.runs:
            run.font.bold = True
            
# Add Data Rows
for index, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        # Format decimal places nicely for presentation
        if isinstance(value, float):
            row_cells[i].text = f"{value:.2f}"
        else:
            row_cells[i].text = str(value)

# Save the document
doc.save(output_word_path)
print(f"\n✅ Successfully exported Chapter 4 table to: {output_word_path}")

--- FORMATTING CHAPTER 4 RESULTS TABLE ---

✅ Successfully exported Chapter 4 table to: C:\Users\PC\Documents\MSc_Enugu_Data\Chapter_4_UHI_LST_Table.docx


In [9]:
import os
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

print("--- GENERATING CHAPTER 4 VALIDATION METRICS DOCUMENT ---")

# Define paths
base_folder = "C:\\Users\\PC\\Documents\\MSc_Enugu_Data\\"
lulc_csv = os.path.join(base_folder, "Enugu_LULC_Metrics.csv")
uhi_csv = os.path.join(base_folder, "Enugu_UHI_Metrics.csv")
output_doc = os.path.join(base_folder, "Enugu_Validation_Metrics_Chapter4.docx")

# Initialize Word Document
doc = Document()
style = doc.styles['Normal']
style.font.name = 'Times New Roman'
style.font.size = Pt(12)

# Chapter Heading
doc.add_heading('4.2 Application and Performance of the CNN-Driven Model (Objective 2)', level=2)
doc.add_paragraph(
    "To evaluate the deep learning approach's accuracy and verify its capacity for high-resolution mapping, "
    "the Hybrid ConvLSTM model was validated against the 2024 ground truth dataset. The performance is quantified "
    "using Overall Accuracy (OA), Cohen’s Kappa Coefficient, Root Mean Square Error (RMSE), and the Coefficient of Determination (R²)."
)

# Overall Metrics Summary Block
doc.add_heading('Overall Performance Metrics', level=3)
p_metrics = doc.add_paragraph()
p_metrics.add_run("LULC Prediction Core:\n").bold = True
p_metrics.add_run("• Overall Accuracy (OA): [Insert OA % here from your console]\n")
p_metrics.add_run("• Kappa Coefficient: [Insert Kappa here]\n")
p_metrics.add_run("• RMSE: [Insert RMSE here]  |  R²: [Insert R2 here]\n\n")

p_metrics.add_run("UHI Prediction Core:\n").bold = True
p_metrics.add_run("• Overall Accuracy (OA): [Insert OA % here from your console]\n")
p_metrics.add_run("• Kappa Coefficient: [Insert Kappa here]\n")
p_metrics.add_run("• RMSE: [Insert RMSE here]  |  R²: [Insert R2 here]")

doc.add_paragraph(
    "\nThe detailed class-by-class performance, including Producer’s Accuracy (PA), User’s Accuracy (UA), "
    "F1-Score, and Intersection over Union (IoU), are presented in the tables below."
)

# Helper function to generate formatted tables
def add_metrics_table(document, csv_path, table_title):
    if not os.path.exists(csv_path):
        document.add_paragraph(f"[Error: {os.path.basename(csv_path)} not found in directory]")
        return

    df = pd.read_csv(csv_path)
    
    # Add Table Title
    heading = document.add_paragraph()
    heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
    heading.add_run(table_title).bold = True

    # Create Table
    table = document.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    table.autofit = True
    
    # Add Headers
    hdr_cells = table.rows[0].cells
    for i, col_name in enumerate(df.columns):
        hdr_cells[i].text = str(col_name)
        for paragraph in hdr_cells[i].paragraphs:
            for run in paragraph.runs:
                run.font.bold = True

    # Add Data
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            if isinstance(value, float):
                row_cells[i].text = f"{value:.4f}"
            else:
                row_cells[i].text = str(value)
                
    document.add_paragraph("\n")

# Insert Tables
add_metrics_table(doc, lulc_csv, "Table 2.1: LULC Classification Accuracy Metrics (2024 Validation)")
add_metrics_table(doc, uhi_csv, "Table 2.2: UHI Thermal Zone Classification Accuracy Metrics (2024 Validation)")

# Save Document
doc.save(output_doc)
print(f"✅ Successfully exported validation metrics to: {output_doc}")

--- GENERATING CHAPTER 4 VALIDATION METRICS DOCUMENT ---
✅ Successfully exported validation metrics to: C:\Users\PC\Documents\MSc_Enugu_Data\Enugu_Validation_Metrics_Chapter4.docx
